# Gradient descent, one step at a time

Gradient descent is often summarized as “move downhill.” In this activity, you will make that phrase precise. You will work with **one parameter**, $\theta$, so every update can be calculated, plotted, and explained.

By the end, you should be able to:

1. connect the sign and size of a derivative to the direction and length of an update;
2. translate $\theta_{t+1}=\theta_t-\eta f'(\theta_t)$ into a few lines of Python;
3. explain how the learning rate changes convergence;
4. use an optimization trace to diagnose slow progress, oscillation, and divergence; and
5. explain why a small gradient does not always identify the global minimum.

Write your predictions before running each experiment; a correct prediction is useful, but a revised prediction is often more instructive.

## Activity map

| Part | Main idea | What you produce |
|---|---|---|
| 1. Read a slope | A derivative is local information | Direction predictions |
| 2. Take one step | The update rule combines slope and learning rate | One hand-coded update |
| 3. Build the algorithm | Repetition creates a trajectory | Your own gradient-descent function |
| 4. Read a trace | Saved states make behavior visible | A trace-based explanation |
| 5. Stress-test $\eta$ | Step size controls behavior | A learning-rate diagnosis |
| 6. Decide when to stop | “Converged” needs an operational definition | A stopping rule |
| 7. Leave the bowl | Starting values matter on nonconvex objectives | A qualified conclusion |
| 8. Exit ticket | Connect code, mathematics, and interpretation | An individual summary |

This notebook contains **Try it** cells and collapsible hints. The supplied reference functions let later sections run even if your own version is unfinished. Do not open a hint until you have written or discussed an attempt.

## 0. Setup

Only NumPy and Matplotlib are needed; the data and objectives are generated locally.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {
    "start": "#8E44AD",
    "path": "#E67E22",
    "minimum": "#168A75",
    "curve": "#2C3E50",
}
np.set_printoptions(precision=4, suppress=True)

## 1. Read the slope before moving

We begin with a bowl-shaped objective:

$$
f(\theta)=\frac{1}{2}(\theta-3)^2+1,
\qquad
f'(\theta)=\theta-3.
$$

The minimum is at $\theta^*=3$. The optimizer is allowed to evaluate $f$ and $f'$, but imagine that it does **not** know $\theta^*$.

### Predict 1

Without running code, fill in the last two columns.

| Current $\theta$ | Sign of $f'(\theta)$ | Does $-f'(\theta)$ point left or right? | Should one descent step increase or decrease $\theta$? |
|---:|:---:|:---:|:---:|
| $-2$ |  |  |  |
| $2$ |  |  |  |
| $3$ |  |  |  |
| $5$ |  |  |  |

**Reason in words:** Why does the update use the *negative* derivative?

In [ ]:
# Both functions accept theta, the model parameter, and return information at that value.

def objective(theta):
    '''A univariate objective with its minimum at theta = 3.'''
    return 0.5 * (theta - 3.0) ** 2 + 1.0


def gradient(theta):
    '''Derivative of objective(theta).'''
    return theta - 3.0


theta_grid = np.linspace(-5, 7, 400)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(theta_grid, objective(theta_grid), color=COLORS["curve"], linewidth=2.5)
ax.scatter([3], [objective(3)], color=COLORS["minimum"], s=90, zorder=3, label=r"minimum $\theta^*=3$")
for theta in [-2, 2, 3, 5]:
    ax.scatter(theta, objective(theta), s=55, zorder=3)
    ax.annotate(f"θ={theta}", (theta, objective(theta)), xytext=(5, 8), textcoords="offset points")
ax.set(xlabel=r"parameter $\theta$", ylabel=r"objective $f(\theta)$", title="The objective landscape")
ax.legend()
plt.show()

In [ ]:
# Check the signs after you make your predictions.
probe_points = np.array([-2.0, 2.0, 3.0, 5.0])
for theta in probe_points:
    direction = "right" if -gradient(theta) > 0 else "left" if -gradient(theta) < 0 else "stay"
    print(f"theta={theta:>4.1f} | gradient={gradient(theta):>5.1f} | -gradient points {direction}")

### Checkpoint 1

The derivative is a **local slope**, not the location of the minimum. A positive derivative means $f$ increases as $\theta$ moves right, so $-f'(\theta)$ points left. A negative derivative makes the descent direction point right. At $\theta=3$, the derivative is zero and the update has no reason to move.

Notice one limitation already: $f'(\theta)=0$ says “stationary,” not automatically “global minimum.” We will return to this in Part 7.

## 2. Take exactly one gradient-descent step

The update rule is

$$
\underbrace{\theta_{t+1}}_{\text{new position}}
=
\underbrace{\theta_t}_{\text{current position}}
-
\underbrace{\eta}_{\text{learning rate}}
\underbrace{f'(\theta_t)}_{\text{local slope}}.
$$

Start at $\theta_0=-4$ and use $\eta=0.25$.

### Predict 2

Before coding:

1. Is the first step left or right?
2. What are $f'(\theta_0)$ and $\theta_1$?
3. Should $f(\theta_1)$ be smaller than $f(\theta_0)$?

In [ ]:
# TRY IT: replace None with the update expression.
theta_0 = -4.0
learning_rate = 0.25

theta_1 = None

if theta_1 is None:
    print("Replace theta_1 = None with your update expression, then rerun this cell.")
else:
    expected = theta_0 - learning_rate * gradient(theta_0)
    print(f"Your theta_1: {theta_1:.4f}")
    print("Update check:", "passed" if np.isclose(theta_1, expected) else "not yet")
    print("Objective decreased:", objective(theta_1) < objective(theta_0))

<details>
<summary><strong>Hint</strong></summary>

Substitute the current point into the derivative first, then multiply by the learning rate.

</details>

In [ ]:
def plot_one_step(theta, learning_rate, ax=None):
    '''Show the tangent information and the resulting scalar update.'''
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4.8))

    theta_next = theta - learning_rate * gradient(theta)
    x = np.linspace(-5, 7, 400)
    tangent_x = np.linspace(theta - 1.4, theta + 1.4, 60)
    tangent_y = objective(theta) + gradient(theta) * (tangent_x - theta)

    ax.plot(x, objective(x), color=COLORS["curve"], linewidth=2.3, label=r"$f(\theta)$")
    ax.plot(tangent_x, tangent_y, "--", color="#7F8C8D", label="local tangent")
    ax.scatter(theta, objective(theta), color=COLORS["start"], s=85, zorder=5, label="current")
    ax.scatter(theta_next, objective(theta_next), color=COLORS["path"], s=85, zorder=5, label="next")
    ax.axvline(3, color=COLORS["minimum"], linestyle=":", linewidth=2, label=r"$\theta^*=3$")
    ax.annotate(
        "",
        xy=(theta_next, objective(theta_next)),
        xytext=(theta, objective(theta)),
        arrowprops={"arrowstyle": "->", "color": COLORS["path"], "lw": 2.3},
    )
    ax.set(
        xlabel=r"$\theta$",
        ylabel=r"$f(\theta)$",
        title=fr"One update: $\theta$ {theta:.2f} → {theta_next:.2f}  ($\eta={learning_rate}$)",
    )
    ax.legend(loc="upper right", ncol=2)
    return theta_next


theta_1_reference = plot_one_step(theta=-4.0, learning_rate=0.25)
plt.show()
print(f"Reference update: theta_1 = {theta_1_reference:.4f}")

## 3. Build gradient descent yourself

The finished demo above shows one update. Now turn that update into an algorithm in small, testable pieces. Work from top to bottom, and replace each `None` or `pass` only after you can describe the corresponding mathematical operation.

| Checkpoint | Your code should do | Evidence to look for |
|---|---|---|
| 1. Direction | Negate the current gradient | The direction points toward $\theta^*=3$ |
| 2. Update | Move by learning rate times direction | The objective decreases for this small step |
| 3. Trace | Save the initial value and every new value | The trace has 9 values |
| 4. Algorithm | Put the repeated update in a function | The final value matches the expected trace |

Every starter cell is safe to run before you finish it, so the rest of the notebook remains runnable. The checks are deliberately small: they test behavior, not a particular coding style.

In [ ]:
# Build 1 of 4: compute the descent direction.
diy_theta = -4.0
diy_rate = 0.25
descent_direction = None  # Replace with an expression involving gradient(diy_theta).

if descent_direction is None:
    print("Write descent_direction = ... and rerun this cell.")

In [ ]:
# Build 2 of 4: turn the direction into one parameter update.
diy_theta_next = None  # Combine diy_theta, diy_rate, and descent_direction.

if diy_theta_next is None:
    print("Write diy_theta_next = ... and rerun this cell.")

In [ ]:
# Build 3 of 4: repeat the update and record every state.
trace_theta = -4.0  # Reset the start so this cell is safe to rerun.
trace_rate = 0.25
diy_trace = [trace_theta]

for _ in range(8):
    # Replace the next two lines with your update and append operations.
    # trace_theta = ...
    # diy_trace.append(...)
    pass

if len(diy_trace) == 1:
    print("Add the update and append lines, then rerun.")


In [ ]:
# Build 4 of 4: package the repeated process as a reusable function.
def diy_gradient_descent(theta0, learning_rate, n_steps):
    theta = float(theta0)
    trace = [theta]
    for _ in range(n_steps):
        # Replace the next two lines with the same update-and-record logic.
        # theta = ...
        # trace.append(...)
        pass
    return trace

diy_function_trace = diy_gradient_descent(-4.0, 0.25, 8)
expected_function_final = 3 + (-4 - 3) * (1 - 0.25) ** 8
if len(diy_function_trace) == 1:
    print("Complete the function body, then rerun.")
else:
    print("Function check:", "passed" if len(diy_function_trace) == 9 else "not yet")
    print("Final-value check:", "passed" if np.isclose(diy_function_trace[-1], expected_function_final) else "not yet")

### Build reflection

Before moving on, point to the line in your function that does each job: evaluates the gradient, scales the step, updates the parameter, and records the new state. What would go wrong if the gradient were evaluated once before the loop instead of at the current value on every iteration?

### Explore one-step behavior

For each combination below, predict whether the step moves toward the minimizer, crosses it, and decreases the objective. Then run the next cell and check all three predictions.

**Explain:** A step may point downhill locally and still increase the objective after the update. How can both statements be true?

In [ ]:
one_step_experiments = [
    (5.0, 0.25),
    (-4.0, 1.25),
    (5.0, 2.10),
]

for start, rate in one_step_experiments:
    plot_one_step(theta=start, learning_rate=rate)
    plt.show()

## 4. Read a diagnostic trace

Your DIY function saves the sequence of parameter values. A diagnostic **trace** keeps more context: the step number, current parameter, objective value, current gradient, and the update that will produce the next row. The final row has no next update, so that entry is blank.

Before running the reference implementation below, discuss two questions:

1. Why is the initial state part of the trace?
2. Which saved quantity would reveal each behavior: slow progress, oscillation, and divergence?
3. If your DIY function is complete, how should its nine values relate to the `theta` column below?

In [ ]:
def gradient_descent(objective_fn, gradient_fn, theta0, learning_rate, n_steps):
    '''Run scalar gradient descent and record a diagnostic trace.'''
    theta = float(theta0)
    history = []

    for step in range(n_steps):
        grad = float(gradient_fn(theta))
        update = -learning_rate * grad
        history.append({
            "step": step,
            "theta": theta,
            "objective": float(objective_fn(theta)),
            "gradient": grad,
            "update": update,
        })
        theta = theta + update

    # Record the final state; there is no update after this row.
    history.append({
        "step": n_steps,
        "theta": theta,
        "objective": float(objective_fn(theta)),
        "gradient": float(gradient_fn(theta)),
        "update": np.nan,
    })
    return history


history = gradient_descent(objective, gradient, theta0=-4, learning_rate=0.25, n_steps=8)
print(f"{'step':>4} {'theta':>10} {'gradient':>10} {'next update':>12} {'objective':>11}")
for row in history:
    update_text = "--" if np.isnan(row["update"]) else f'{row["update"]:.4f}'
    print(f'{row["step"]:4d} {row["theta"]:10.4f} {row["gradient"]:10.4f} {update_text:>12} {row["objective"]:11.4f}')

### Read the trace

Here, `history` is the trace: each row is a snapshot at one update number. In every row except the last, `update` is the change that takes `theta` to the next row. The left plot shows where the optimizer is in parameter space; the right plot shows how the objective gap changes over time.

Use the table above to answer:

1. Which quantities shrink as the optimizer approaches $\theta^*=3$?
2. The learning rate stays fixed. Why does the update length still shrink?
3. Verify from one row that `update` equals `-learning_rate * gradient`.
4. Why are both the step-0 row and the final row useful?

In [ ]:
def plot_trajectory(history, objective_fn, xlim=(-5, 7), title="Gradient-descent trajectory"):
    thetas = np.array([row["theta"] for row in history])
    values = np.array([row["objective"] for row in history])
    steps = np.array([row["step"] for row in history])
    x = np.linspace(*xlim, 500)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    axes[0].plot(x, objective_fn(x), color=COLORS["curve"], linewidth=2.3)
    axes[0].plot(thetas, values, "o-", color=COLORS["path"], linewidth=1.7, markersize=5)
    axes[0].scatter(thetas[0], values[0], color=COLORS["start"], s=95, zorder=5, label="start")
    axes[0].scatter(thetas[-1], values[-1], color=COLORS["minimum"], s=95, zorder=5, label="final")
    for i in range(min(5, len(thetas) - 1)):
        axes[0].annotate("", xy=(thetas[i+1], values[i+1]), xytext=(thetas[i], values[i]),
                         arrowprops={"arrowstyle": "->", "color": COLORS["path"], "lw": 1.8})
        axes[0].annotate(str(i), (thetas[i], values[i]), xytext=(3, 6), textcoords="offset points")
    axes[0].axvline(3, color=COLORS["minimum"], linestyle=":", linewidth=1.8)
    axes[0].set(xlabel=r"$\theta$", ylabel=r"$f(\theta)$", title=title)
    axes[0].legend()

    axes[1].semilogy(steps, values - 1.0, "o-", color=COLORS["path"])
    axes[1].set(xlabel="update number", ylabel=r"objective gap $f(\theta_t)-f(\theta^*)$",
                title="Progress over iterations")
    fig.tight_layout()
    return fig, axes


plot_trajectory(history, objective, title=r"Each update moves toward $\theta^*=3$")
plt.show()

In [ ]:
# A frame-by-frame view of the first six updates.
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)
for t, ax in enumerate(axes.flat):
    plot_one_step(history[t]["theta"], learning_rate=0.25, ax=ax)
    ax.set_title(f"Update {t} → {t+1}")
    ax.legend().remove()
fig.suptitle("The slope and step both shrink near the minimizer", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

## 5. The learning rate changes the story

For this objective, the distance from the minimum follows a particularly useful recurrence:

$$
\theta_{t+1}-3=(1-\eta)(\theta_t-3).
$$

### Predict 5

Before running the comparison, predict the behavior of each rate.

| $\eta$ | Slow approach, fast approach, oscillation, or divergence? | Will it cross $\theta^*=3$? |
|---:|:---|:---:|
| 0.10 |  |  |
| 0.80 |  |  |
| 1.80 |  |  |
| 2.10 |  |  |

Tip: focus on the magnitude and sign of $1-\eta$.

In [ ]:
rates = [0.10, 0.80, 1.80, 2.10]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.7))

for rate in rates:
    run = gradient_descent(objective, gradient, theta0=-4, learning_rate=rate, n_steps=18)
    steps = np.array([row["step"] for row in run])
    thetas = np.array([row["theta"] for row in run])
    # This equivalent formula avoids rounding tiny positive gaps down to zero.
    gaps = 0.5 * (thetas - 3.0) ** 2
    axes[0].plot(steps, thetas, "o-", markersize=3.5, label=fr"$\eta={rate}$")
    axes[1].semilogy(steps, gaps, "o-", markersize=3.5, label=fr"$\eta={rate}$")

axes[0].axhline(3, color=COLORS["minimum"], linestyle=":", linewidth=2, label=r"$\theta^*=3$")
axes[0].set(xlabel="update number", ylabel=r"$\theta_t$", title="Parameter trace")
axes[1].set(xlabel="update number", ylabel="objective gap", title="Objective trace (log scale)")
axes[0].legend(ncol=2)
axes[1].legend(ncol=2)
fig.tight_layout()
plt.show()

### Diagnose, do not just label

For each curve, cite visible evidence from **both** panels.

- **Slow progress:** What changes steadily but only a little?
- **Fast convergence:** Which rate removes most of the error in a few updates?
- **Stable oscillation:** Which trace alternates around 3 while its distance shrinks?
- **Divergence:** Which objective gap grows?

For this particular quadratic, convergence requires $|1-\eta|<1$, or $0<\eta<2$. That interval is not a universal learning-rate rule; it follows from this objective's curvature.

### Experiment 5 — Find a fast, non-crossing rate

Start at $\theta_0=-4$. Find a learning rate that reaches $|\theta_t-3|<0.001$ in at most 10 updates **without ever crossing 3**.

1. Try values by editing `candidate_rate`.
2. Use the feedback, not trial count alone, to refine your reasoning.
3. Explain your choice using $1-\eta$.

In [ ]:
candidate_rate = 0.25  # Change this.
candidate_run = gradient_descent(objective, gradient, theta0=-4, learning_rate=candidate_rate, n_steps=10)
candidate_thetas = np.array([row["theta"] for row in candidate_run])

reached = np.any(np.abs(candidate_thetas - 3) < 0.001)
crossed = np.any(candidate_thetas > 3)
finite = np.all(np.isfinite(candidate_thetas))

print(f"Final theta: {candidate_thetas[-1]:.6f}")
print("Reached tolerance within 10 updates:", reached)
print("Crossed the minimizer:", crossed)
print("Trace remained finite:", finite)

<details>
<summary><strong>Reasoning hint</strong></summary>

To avoid crossing from a starting point left of 3, choose $0<\eta\leq 1$. The remaining distance is multiplied by $|1-\eta|$ each time. Rates close to 1 remove the distance quickly; $\eta=1$ lands exactly at the minimum in one update for this special quadratic.
</details>

## 6. “Stop” must mean something measurable

A fixed number of updates is simple but may waste work or stop too early. Common alternatives stop when:

- the gradient magnitude is small: $|f'(\theta_t)|<\varepsilon$;
- the update magnitude is small: $|\eta f'(\theta_t)|<\varepsilon$; or
- the objective changes very little.

These criteria are related but not identical. A tiny learning rate can make the update tiny even when the gradient is not.

### Predict 6

If $|\text{update}|<10^{-4}$ is the only rule, why could using $\eta=10^{-8}$ produce a misleading “converged” message far from the minimizer?

In [ ]:
def gradient_descent_until(
    objective_fn,
    gradient_fn,
    theta0,
    learning_rate,
    gradient_tolerance=1e-4,
    max_steps=10_000,
):
    '''Scalar gradient descent with a gradient-based stopping rule.'''
    theta = float(theta0)
    history = []

    for step in range(max_steps + 1):
        grad = float(gradient_fn(theta))
        history.append((step, theta, float(objective_fn(theta)), grad))
        if abs(grad) < gradient_tolerance:
            return theta, history, "gradient tolerance reached"
        if step == max_steps:
            return theta, history, "maximum steps reached"
        theta = theta - learning_rate * grad


for rate in [0.05, 0.25, 0.80]:
    theta_hat, run, reason = gradient_descent_until(
        objective, gradient, theta0=-4, learning_rate=rate, gradient_tolerance=1e-4
    )
    print(f"eta={rate:>4.2f} | updates={len(run)-1:>4d} | theta={theta_hat:.6f} | {reason}")

### Checkpoint 6

Answer in two or three sentences:

1. Why do the three rates stop after different numbers of updates even though the tolerance is the same?
2. What protection does `max_steps` provide?
3. Would a small gradient alone prove that the **global** minimum was found? Keep your answer tentative until Part 7.

## 7. Leave the friendly bowl

Now consider another univariate objective:

$$
q(\theta)=0.08\theta^4-0.6\theta^2+0.25\theta+2,
$$

with derivative

$$
q'(\theta)
$$

This objective has more than one valley. Nothing about the update rule changes, but the conclusion we can draw from convergence does.

### Predict 7

Inspect the landscape first. Mark approximate locations of every local minimum and maximum. Then predict where runs starting at $-2.4$, $0$, and $2.4$ will finish.

In [ ]:
def nonconvex_objective(theta):
    return 0.08 * theta**4 - 0.6 * theta**2 + 0.25 * theta + 2.0


# Write the derivative function here.
def nonconvex_gradient(theta):
    # Derivative of nonconvex_objective(theta).
    pass


x = np.linspace(-3, 3, 600)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, nonconvex_objective(x), color=COLORS["curve"], linewidth=2.4)
for start in [-2.4, 0.0, 2.4]:
    ax.scatter(start, nonconvex_objective(start), s=75, label=fr"start $\theta_0={start}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$q(\theta)$", title="A nonconvex univariate objective")
ax.legend(ncol=3)
plt.show()

In [ ]:
starts = [-2.4, 0.0, 2.4]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, nonconvex_objective(x), color=COLORS["curve"], linewidth=2.4, label=r"$q(\theta)$")
print(f"{'start':>7} {'final theta':>12} {'final q(theta)':>15} {'|final gradient|':>17}")

for start in starts:
    run = gradient_descent(
        nonconvex_objective, nonconvex_gradient,
        theta0=start, learning_rate=0.12, n_steps=50
    )
    thetas = np.array([row["theta"] for row in run])
    values = np.array([row["objective"] for row in run])
    final_gradient = abs(nonconvex_gradient(thetas[-1]))
    print(f"{start:7.1f} {thetas[-1]:12.6f} {values[-1]:15.6f} {final_gradient:17.2e}")
    ax.plot(thetas, values, "o-", markersize=3.5, markevery=3, linewidth=1.5, label=fr"start {start} → {thetas[-1]:.3f}")

ax.set(xlabel=r"$\theta$", ylabel=r"$q(\theta)$", title="Same update rule, different starting values")
ax.legend(ncol=2)
plt.show()

### Synthesize

1. Which starting values reached the same valley? Which reached a different one?
2. Did the runs that ended in different valleys use different learning rates or update rules?
3. Compare the final objective values. Did every run find the global minimum?
4. Revise this statement so it is defensible: **“Gradient descent converged, so it found the best possible parameter.”**

<details>
<summary><strong>Interpretation check</strong></summary>

A defensible version is: “From this starting value and learning rate, gradient descent reached a stationary point with a small gradient.” On a nonconvex objective, that statement alone does not establish global optimality.
</details>

## 8. Exit ticket

Answer without running new code.

1. For another differentiable objective, the current value is $\theta=5$ and the derivative there is $-4$. With $\eta=0.1$, what is the next value of $\theta$? Which direction does it move?
2. Two runs use the same starting value. Run A decreases smoothly but slowly; Run B alternates across a minimum while its objective decreases. Which run likely has the larger learning rate? What evidence supports your answer?
3. Name two pieces of information you would save in an optimization trace and explain what each helps diagnose.
4. In one sentence, distinguish a **stationary point** from a **global minimum**.
5. Write the core update from memory in both mathematical notation and Python.

### Takeaway

Gradient descent does not jump to a known minimizer. It repeatedly uses local slope information. The derivative selects a direction and contributes a magnitude; the learning rate scales that move; the objective landscape and starting value determine what “downhill” can ultimately reach.

## Optional extension — Design your own univariate objective

Create a differentiable function and its derivative. Before running the optimizer:

1. sketch or plot the objective;
2. predict the effect of two starting values and two learning rates;
3. run `gradient_descent`;
4. decide whether the result supports your prediction; and
5. verify your derivative numerically using

$$
f'(\theta)\approx \frac{f(\theta+h)-f(\theta-h)}{2h}
$$

for a small $h$ such as $10^{-5}$.

In [ ]:
# OPTIONAL STARTER
def my_objective(theta):
    return (theta + 1) ** 2  # Replace with your function.


def my_gradient(theta):
    return 2 * (theta + 1)   # Replace with its derivative.


test_theta = 0.7
h = 1e-5
numerical_gradient = (my_objective(test_theta + h) - my_objective(test_theta - h)) / (2 * h)
print("Analytic derivative:", my_gradient(test_theta))
print("Numerical check:   ", numerical_gradient)
print("Close:", np.isclose(my_gradient(test_theta), numerical_gradient, rtol=1e-5, atol=1e-7))